# Transformando los datos 

In [1]:
from pathlib import Path
import pandas as pd

CARPETA = Path(r"D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME")
df_2024 = pd.read_csv(CARPETA / "enaho_mod1_2024_filtrado.csv")
df_2025 = pd.read_csv(CARPETA / "enaho_mod1_2025_filtrado.csv")

# Crear target: 0 = respuesta (result es 1 o 2), 1 = no respuesta (result no es 1 ni 2)
for df in [df_2024, df_2025]:
    df["target"] = (~df["result"].isin([1, 2])).astype(int)

tabla_2024 = (df_2024["target"].value_counts(normalize=True) * 100).round(2).rename("2024")
tabla_2025 = (df_2025["target"].value_counts(normalize=True) * 100).round(2).rename("2025")

tabla = pd.concat([tabla_2024, tabla_2025], axis=1).sort_index()
tabla.index = tabla.index.map({0: "Respuesta", 1: "No respuesta"})
tabla.index.name = "Categoría"
tabla.columns = ["% 2024", "% 2025"]

# Tabla con estilo (sin necesidad de matplotlib)
tabla.style.format("{:.2f}%").set_caption("Distribución de Target por Año").set_table_styles([
    {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold")]},
    {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"), ("text-align", "center"), ("padding", "8px")]},
    {"selector": "td", "props": [("text-align", "center"), ("padding", "8px")]},
]).set_properties(**{"border": "1px solid #ddd"})

,% 2024,% 2025
Categoría,,
Respuesta,75.32%,75.57%
No respuesta,24.68%,24.43%


In [2]:
from pathlib import Path
import pandas as pd

# Crear variables de departamento y provincia a partir del ubigeo
CARPETA = Path(r"D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME")
df_2024 = pd.read_csv(CARPETA / "enaho_mod1_2024_filtrado.csv")
df_2025 = pd.read_csv(CARPETA / "enaho_mod1_2025_filtrado.csv")

for df in [df_2024, df_2025]:
    # target (del paso anterior)
    df["target"] = (~df["result"].isin([1, 2])).astype(int)
    # ubigeo viene como entero -> hay que rellenar con ceros a la izquierda
    # para que tenga siempre 6 dígitos (departamento+provincia+distrito)
    ubigeo_str = df["ubigeo"].astype(str).str.zfill(6)
    df["departamento"] = ubigeo_str.str[0:2]   # primeros 2 dígitos
    df["provincia"] = ubigeo_str.str[0:4]      # primeros 4 dígitos (dep+prov)

# Diccionario de código -> nombre de departamento (INEI)
NOMBRES_DEPARTAMENTO = {
    "01": "Amazonas", "02": "Áncash", "03": "Apurímac", "04": "Arequipa",
    "05": "Ayacucho", "06": "Cajamarca", "07": "Callao", "08": "Cusco",
    "09": "Huancavelica", "10": "Huánuco", "11": "Ica", "12": "Junín",
    "13": "La Libertad", "14": "Lambayeque", "15": "Lima", "16": "Loreto",
    "17": "Madre de Dios", "18": "Moquegua", "19": "Pasco", "20": "Piura",
    "21": "Puno", "22": "San Martín", "23": "Tacna", "24": "Tumbes",
    "25": "Ucayali",
}

for df in [df_2024, df_2025]:
    df["nombre_departamento"] = df["departamento"].map(NOMBRES_DEPARTAMENTO)

# Vista previa de las variables creadas
print("Vista previa 2024:")
print(df_2024[["ubigeo", "departamento", "nombre_departamento", "provincia"]].head())

# Cuadro resumen: hogares por departamento, 2024 vs 2025
resumen_2024 = df_2024["nombre_departamento"].value_counts().rename("n_2024")
resumen_2025 = df_2025["nombre_departamento"].value_counts().rename("n_2025")

tabla_dep = pd.concat([resumen_2024, resumen_2025], axis=1).fillna(0).astype(int)
tabla_dep = tabla_dep.sort_values("n_2024", ascending=False)
tabla_dep.index.name = "Departamento"
tabla_dep.columns = ["Hogares 2024", "Hogares 2025"]

# Fila de totales y de conteo de unidades geográficas
tabla_dep.loc["Total"] = tabla_dep.sum()

print(f"\nDepartamentos únicos 2024: {df_2024['departamento'].nunique()} | "
      f"Provincias únicas 2024: {df_2024['provincia'].nunique()}")
print(f"Departamentos únicos 2025: {df_2025['departamento'].nunique()} | "
      f"Provincias únicas 2025: {df_2025['provincia'].nunique()}\n")

# Tabla con estilo (sin dependencias externas)
tabla_dep.style.format("{:,.0f}").set_caption("Hogares encuestados por Departamento").set_table_styles([
    {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold")]},
    {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"), ("text-align", "center"), ("padding", "6px")]},
    {"selector": "td", "props": [("text-align", "center"), ("padding", "6px")]},
]).set_properties(**{"border": "1px solid #ddd"})


Vista previa 2024:
   ubigeo departamento nombre_departamento provincia
0   10101           01            Amazonas      0101
1   10101           01            Amazonas      0101
2   10101           01            Amazonas      0101
3   10101           01            Amazonas      0101
4   10101           01            Amazonas      0101

Departamentos únicos 2024: 25 | Provincias únicas 2024: 196
Departamentos únicos 2025: 25 | Provincias únicas 2025: 196



,Hogares 2024,Hogares 2025
Departamento,,
Lima,"6,174","6,223"
Arequipa,"2,144","2,123"
La Libertad,"1,961","1,952"
Piura,"1,935","1,918"
Puno,"1,883","1,949"
Junín,"1,880","1,899"
Tacna,"1,879","1,929"
Áncash,"1,845","1,808"
Cajamarca,"1,841","1,822"


In [3]:
from pathlib import Path
import pandas as pd

CARPETA = Path(r"D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME")
df_2024 = pd.read_csv(CARPETA / "enaho_mod1_2024_filtrado.csv")
df_2025 = pd.read_csv(CARPETA / "enaho_mod1_2025_filtrado.csv")

for df in [df_2024, df_2025]:
    # target (paso 1)
    df["target"] = (~df["result"].isin([1, 2])).astype(int)
    # departamento y provincia (paso 2)
    ubigeo_str = df["ubigeo"].astype(str).str.zfill(6)
    df["departamento"] = ubigeo_str.str[0:2]
    df["provincia"] = ubigeo_str.str[0:4]
    # urbano/rural (paso 3): 1-5 = Urbano, 6+ = Rural
    df["area"] = df["estrato"].apply(lambda x: "Urbano" if x <= 5 else "Rural")

# Cuadro comparativo urbano/rural, 2024 vs 2025
tabla_2024 = (df_2024["area"].value_counts(normalize=True) * 100).round(2).rename("2024")
tabla_2025 = (df_2025["area"].value_counts(normalize=True) * 100).round(2).rename("2025")

tabla_area = pd.concat([tabla_2024, tabla_2025], axis=1).sort_index(ascending=False)  # Urbano arriba
tabla_area.index.name = "Área"
tabla_area.columns = ["% 2024", "% 2025"]
tabla_area.loc["Total"] = tabla_area.sum()

# Tabla con estilo (sin dependencias externas)
tabla_area.style.format("{:.2f}%").set_caption("Distribución Urbano / Rural por Año").set_table_styles([
    {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold")]},
    {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"), ("text-align", "center"), ("padding", "8px")]},
    {"selector": "td", "props": [("text-align", "center"), ("padding", "8px")]},
]).set_properties(**{"border": "1px solid #ddd"})

,% 2024,% 2025
Área,,
Urbano,65.99%,66.06%
Rural,34.01%,33.94%
Total,100.00%,100.00%


In [4]:
from pathlib import Path
import pandas as pd

CARPETA = Path(r"D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME")
df_2024 = pd.read_csv(CARPETA / "enaho_mod1_2024_filtrado.csv")
df_2025 = pd.read_csv(CARPETA / "enaho_mod1_2025_filtrado.csv")

# Mapeo oficial de INEI para 'dominio' (diccionario ENAHO):
# 1 Costa Norte, 2 Costa Centro, 3 Costa Sur,
# 4 Sierra Norte, 5 Sierra Centro, 6 Sierra Sur,
# 7 Selva, 8 Lima Metropolitana
mapa_region = {
    1: "Costa",
    2: "Costa",
    3: "Costa",
    4: "Sierra",
    5: "Sierra",
    6: "Sierra",
    7: "Selva",
    8: "Lima Metropolitana",
}

for df in [df_2024, df_2025]:
    # target (paso 1)
    df["target"] = (~df["result"].isin([1, 2])).astype(int)
    # departamento y provincia (paso 2)
    ubigeo_str = df["ubigeo"].astype(str).str.zfill(6)
    df["departamento"] = ubigeo_str.str[0:2]
    df["provincia"] = ubigeo_str.str[0:4]
    # urbano/rural (paso 3)
    df["area"] = df["estrato"].apply(lambda x: "Urbano" if x <= 5 else "Rural")
    # región natural (paso 4)
    df["region_natural"] = df["dominio"].map(mapa_region)

# Cuadro comparativo de región natural, 2024 vs 2025
tabla_2024 = (df_2024["region_natural"].value_counts(normalize=True) * 100).round(2).rename("2024")
tabla_2025 = (df_2025["region_natural"].value_counts(normalize=True) * 100).round(2).rename("2025")

orden = ["Costa", "Sierra", "Selva", "Lima Metropolitana"]
tabla_region = pd.concat([tabla_2024, tabla_2025], axis=1).reindex(orden)
tabla_region.index.name = "Región Natural"
tabla_region.columns = ["% 2024", "% 2025"]
tabla_region.loc["Total"] = tabla_region.sum()

# Tabla con estilo (sin dependencias externas)
tabla_region.style.format("{:.2f}%").set_caption("Distribución por Región Natural, 2024 vs 2025").set_table_styles([
    {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold")]},
    {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"), ("text-align", "center"), ("padding", "8px")]},
    {"selector": "td", "props": [("text-align", "center"), ("padding", "8px")]},
]).set_properties(**{"border": "1px solid #ddd"})

,% 2024,% 2025
Región Natural,,
Costa,29.61%,29.86%
Sierra,37.27%,37.12%
Selva,20.65%,20.52%
Lima Metropolitana,12.46%,12.50%
Total,99.99%,100.00%


In [5]:
#Crea variables discretas para cada mes y trimestre, así como para las variables de departamento, provincia, urbano, rural, región natural y dominio.

CARPETA = Path(r"D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME")

df_2024 = pd.read_csv(CARPETA / "enaho_mod1_2024_filtrado.csv")
df_2025 = pd.read_csv(CARPETA / "enaho_mod1_2025_filtrado.csv")

mapa_region = {1:"Costa", 2:"Costa", 3:"Costa", 4:"Sierra", 5:"Sierra", 6:"Sierra", 7:"Selva", 8:"Lima Metropolitana"}

dataframes = {}

for año, df in [(2024, df_2024), (2025, df_2025)]:
    # target (paso 1)
    df["target"] = (~df["result"].isin([1, 2])).astype(int)

    # departamento y provincia (paso 2)
    ubigeo_str = df["ubigeo"].astype(str).str.zfill(6)
    df["departamento"] = ubigeo_str.str[0:2]
    df["provincia"] = ubigeo_str.str[0:4]

    # urbano/rural (paso 3)
    df["area"] = df["estrato"].apply(lambda x: "Urbano" if x <= 5 else "Rural")

    # región natural (paso 4)
    df["region_natural"] = df["dominio"].map(mapa_region)

    # trimestre a partir de mes (paso 5, parte 1)
    df["trimestre"] = ((df["mes"] - 1) // 3) + 1

    # variables discretas / dummies (paso 5, parte 2)
    variables_a_dummy = ["mes", "trimestre", "departamento", "provincia", "area", "region_natural", "dominio"]
    df_dummies = pd.get_dummies(df, columns=variables_a_dummy, prefix=variables_a_dummy)

    dataframes[año] = df_dummies
    print(f"{año}: shape original {df.shape} -> shape con dummies {df_dummies.shape}")







2024: shape original (44731, 14) -> shape con dummies (44731, 258)
2025: shape original (44599, 14) -> shape con dummies (44599, 258)


In [6]:
#MOSTRAR TABLAS COMPARATIVAS DE VARIABLES CATEGÓRICAS

from pathlib import Path
import pandas as pd
from IPython.display import display

CARPETA = Path(r"D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME")

def transformar(path):
    df = pd.read_csv(path)
    df["target"] = (~df["result"].isin([1, 2])).astype(int)
    ubigeo_str = df["ubigeo"].astype(str).str.zfill(6)
    df["departamento"] = ubigeo_str.str[0:2]
    df["provincia"] = ubigeo_str.str[0:4]
    df["area"] = df["estrato"].apply(lambda x: "Urbano" if x <= 5 else "Rural")
    mapa_region = {1: "Costa", 2: "Costa", 3: "Costa", 4: "Sierra", 5: "Sierra", 6: "Sierra", 7: "Selva", 8: "Lima Metropolitana"}
    df["region_natural"] = df["dominio"].map(mapa_region)
    df["trimestre"] = ((df["mes"] - 1) // 3) + 1
    return df

df_2024 = transformar(CARPETA / "enaho_mod1_2024_filtrado.csv")
df_2025 = transformar(CARPETA / "enaho_mod1_2025_filtrado.csv")

variables_categoricas = ["mes", "trimestre", "departamento", "provincia", "area", "region_natural", "dominio"]

tablas_comparativas = {}

def mostrar_tabla_estilo(tabla, titulo):
    """Muestra un DataFrame con estilo visual tipo reporte."""
    (tabla.style
        .format("{:.2f}%")
        .set_caption(titulo)
        .set_table_styles([
            {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold"), ("text-align", "left"), ("padding-bottom", "6px")]},
            {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"), ("text-align", "center"), ("padding", "6px")]},
            {"selector": "td", "props": [("text-align", "center"), ("padding", "6px")]},
        ])
        .set_properties(**{"border": "1px solid #ddd"})
    )
    display(
        tabla.style
        .format("{:.2f}%")
        .set_caption(titulo)
        .set_table_styles([
            {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold"), ("text-align", "left"), ("padding-bottom", "6px")]},
            {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"), ("text-align", "center"), ("padding", "6px")]},
            {"selector": "td", "props": [("text-align", "center"), ("padding", "6px")]},
        ])
        .set_properties(**{"border": "1px solid #ddd"})
    )

for var in variables_categoricas:
    tabla = pd.DataFrame({
        "2024": df_2024.groupby(var)["target"].mean() * 100,
        "2025": df_2025.groupby(var)["target"].mean() * 100,
    }).round(2)
    tabla.index.name = var.capitalize()
    tabla.columns = ["% 2024", "% 2025"]

    tablas_comparativas[var] = tabla
    mostrar_tabla_estilo(tabla, f"% de no respuesta por {var.capitalize()}")

,% 2024,% 2025
Mes,,
1,24.30%,24.27%
2,24.67%,24.29%
3,23.79%,23.42%
4,25.00%,23.46%
5,24.72%,24.31%
6,22.98%,23.82%
7,25.44%,25.39%
8,25.40%,24.07%
9,25.18%,25.28%


,% 2024,% 2025
Trimestre,,
1,24.25%,23.99%
2,24.25%,23.86%
3,25.34%,24.91%
4,24.88%,24.95%


,% 2024,% 2025
Departamento,,
01,21.48%,21.92%
02,24.50%,24.12%
03,17.30%,16.35%
04,29.76%,28.26%
05,29.61%,29.86%
06,20.21%,19.65%
07,22.90%,22.38%
08,30.31%,30.19%
09,24.87%,23.42%


,% 2024,% 2025
Provincia,,
0101,25.60%,27.05%
0102,21.71%,21.34%
0103,23.00%,16.30%
0104,10.45%,8.73%
0105,18.40%,23.64%
0106,21.79%,24.36%
0107,22.76%,21.58%
0201,22.95%,19.57%
0202,25.00%,33.33%


,% 2024,% 2025
Area,,
Rural,23.37%,23.29%
Urbano,25.36%,25.02%


,% 2024,% 2025
Region_natural,,
Costa,22.09%,21.90%
Lima Metropolitana,26.60%,25.96%
Selva,21.95%,21.36%
Sierra,27.61%,27.66%


,% 2024,% 2025
Dominio,,
1,18.83%,17.59%
2,20.51%,20.76%
3,29.95%,31.13%
4,21.44%,20.69%
5,25.70%,25.87%
6,32.13%,32.30%
7,21.95%,21.36%
8,26.60%,25.96%


In [7]:
#Guarda los dataframes con las nuevas transformaciones

from pathlib import Path
import pandas as pd

# ============================================================
# CONFIGURACIÓN 
# ============================================================
CARPETA_ENTRADA = Path(r"D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME")
CARPETA_SALIDA = Path(r"D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME 2")
AÑOS = [2024, 2025]

MAPA_REGION = {
    1: "Costa", 2: "Costa", 3: "Costa",
    4: "Sierra", 5: "Sierra", 6: "Sierra",
    7: "Selva",
    8: "Lima Metropolitana",
}


# ============================================================
# FUNCIONES
# ============================================================
def transformar(path: Path) -> pd.DataFrame:
    """Lee un dataframe filtrado y aplica todas las transformaciones."""
    df = pd.read_csv(path)

    # 1. target: 0 = respuesta (result 1 o 2), 1 = no respuesta
    df["target"] = (~df["result"].isin([1, 2])).astype(int)

    # 2. departamento y provincia a partir de ubigeo
    ubigeo_str = df["ubigeo"].astype(str).str.zfill(6)
    df["departamento"] = ubigeo_str.str[0:2]
    df["provincia"] = ubigeo_str.str[0:4]

    # 3. urbano/rural a partir de estrato (1-5 urbano, 6+ rural)
    df["area"] = df["estrato"].apply(lambda x: "Urbano" if x <= 5 else "Rural")

    # 4. región natural a partir de dominio
    df["region_natural"] = df["dominio"].map(MAPA_REGION)

    # 5. trimestre a partir de mes
    df["trimestre"] = ((df["mes"] - 1) // 3) + 1

    return df


def main():
    CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
    dataframes_transformados = {}

    for año in AÑOS:
        archivo_entrada = CARPETA_ENTRADA / f"enaho_mod1_{año}_filtrado.csv"

        if not archivo_entrada.exists():
            print(f"⚠️ No se encontró: {archivo_entrada}")
            continue

        df = transformar(archivo_entrada)
        dataframes_transformados[año] = df

        ruta_salida = CARPETA_SALIDA / f"enaho_mod1_{año}_transformado.csv"
        df.to_csv(ruta_salida, index=False, encoding="utf-8")
        print(f"{año}: {df.shape[0]} filas, {df.shape[1]} columnas -> {ruta_salida}")

    return dataframes_transformados


if __name__ == "__main__":
    dataframes_transformados = main()

2024: 44731 filas, 14 columnas -> D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME 2\enaho_mod1_2024_transformado.csv
2025: 44599 filas, 14 columnas -> D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME 2\enaho_mod1_2025_transformado.csv


In [8]:
#GUARDAR LAS TABLAS COMPARATIVAS DE VARIABLES CATEGÓRICAS
from pathlib import Path

CARPETA_SALIDA = Path(r"D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME 2")
CARPETA_TABLAS = CARPETA_SALIDA / "tablas_comparativas"
CARPETA_TABLAS.mkdir(parents=True, exist_ok=True)

for var, tabla in tablas_comparativas.items():
    ruta = CARPETA_TABLAS / f"no_respuesta_por_{var}.csv"
    tabla.to_csv(ruta, encoding="utf-8-sig")
    print(f"Guardado: {ruta}")

Guardado: D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME 2\tablas_comparativas\no_respuesta_por_mes.csv
Guardado: D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME 2\tablas_comparativas\no_respuesta_por_trimestre.csv
Guardado: D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME 2\tablas_comparativas\no_respuesta_por_departamento.csv
Guardado: D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME 2\tablas_comparativas\no_respuesta_por_provincia.csv
Guardado: D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME 2\tablas_comparativas\no_respuesta_por_area.csv
Guardado: D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME 2\tablas_comparativas\no_respuesta_por_region_natural.csv
Guardado: D:\CEU INEI\FUNDAMENTOS DE CIENCIA DE DATO\DATA FRAME 2\tablas_comparativas\no_respuesta_por_dominio.csv
